# RAGAS по реальным вопросам

1. Ключ **OpenAI** в `.env` в корне репозитория (`OPENAI_API_KEY`). Загружается через `load_dotenv`.
2. Вопросы и эталонные ответы: `raw_data/вопрос_ответ_раг.csv` (колонки `question`, `answer`).
3. Ответ и чанки берутся из `query_rag` (ваш пайплайн: ансамбль + реранк + Ollama).

Запускайте ноутбук с **рабочей папкой = корень репозитория** (или оставьте как есть — корень ищется по `mephi_rag/pipeline.py`).

Зависимости:

```bash
pip install ragas langchain-openai openai python-dotenv datasets
```

Параметр `MAX_ROWS` в коде ограничивает число строк (и вызовы к OpenAI на оценку).


In [ ]:
import csv
import sys
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    AnswerRelevancy,
    ContextPrecision,
    Faithfulness,
    LLMContextRecall,
)

REPO = Path.cwd().resolve()
for parent in [REPO, *REPO.parents]:
    if (parent / "mephi_rag" / "pipeline.py").is_file():
        REPO = parent
        break
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

load_dotenv(REPO / ".env")
load_dotenv()

from mephi_rag.pipeline import query_rag

CSV_PATH = REPO / "raw_data" / "вопрос_ответ_раг.csv"
MAX_ROWS = 10

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

rows = []
with CSV_PATH.open(encoding="utf-8", newline="") as f:
    for i, row in enumerate(csv.DictReader(f)):
        if i >= MAX_ROWS:
            break
        question = (row.get("question") or "").strip()
        reference = (row.get("answer") or "").strip()
        if not question or not reference:
            continue
        rag = query_rag(question)
        rows.append(
            {
                "user_input": question,
                "retrieved_contexts": [c["text"] for c in rag["chunks"]],
                "response": (rag["answer"] or "").strip(),
                "reference": reference,
            }
        )

dataset = EvaluationDataset.from_list(rows)

result = evaluate(
    dataset,
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        LLMContextRecall(),
        ContextPrecision(),
    ],
    llm=llm,
    embeddings=embeddings,
)

print(result)
print(result.to_pandas())
